In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        (os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [2]:
!pip install shap -q

In [3]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.applications import ResNet50

import numpy as np
import matplotlib.pyplot as plt
import cv2
import shap

2026-03-18 06:51:06.810835: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1773816667.231194      55 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1773816667.346307      55 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1773816668.297321      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1773816668.297371      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1773816668.297374      55 computation_placer.cc:177] computation placer alr

In [4]:
input_shape = (128,128,3)
batch_size = 16
num_classes = 2

patch_size = 16
projection_dim = 64
num_heads = 4
transformer_layers = 4

epochs = 5

In [5]:
train_ds = tf.keras.preprocessing.image_dataset_from_directory(
    "/kaggle/input/datasets/amimulahasanrofik/alz-b-1100/alzheimer_new_11/alzheimer_new",
    validation_split=0.2,
    subset="training",
    seed=42,
    image_size=(128,128),
    batch_size=batch_size
)

val_ds = tf.keras.preprocessing.image_dataset_from_directory(
    "/kaggle/input/datasets/amimulahasanrofik/alz-b-1100/alzheimer_new_11/alzheimer_new",
    validation_split=0.2,
    subset="validation",
    seed=42,
    image_size=(128,128),
    batch_size=batch_size
)

Found 8800 files belonging to 8 classes.
Using 7040 files for training.


I0000 00:00:1773816708.546378      55 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 15511 MB memory:  -> device: 0, name: Tesla P100-PCIE-16GB, pci bus id: 0000:00:04.0, compute capability: 6.0


Found 8800 files belonging to 8 classes.
Using 1760 files for validation.


In [6]:
AUTOTUNE = tf.data.AUTOTUNE

train_ds = train_ds.prefetch(AUTOTUNE)
val_ds = val_ds.prefetch(AUTOTUNE)

In [7]:
class Patches(layers.Layer):
    def __init__(self, patch_size):
        super().__init__()
        self.patch_size = patch_size

    def call(self, images):
        patches = tf.image.extract_patches(
            images=images,
            sizes=[1,self.patch_size,self.patch_size,1],
            strides=[1,self.patch_size,self.patch_size,1],
            rates=[1,1,1,1],
            padding="VALID"
        )
        patch_dims = patches.shape[-1]
        patches = tf.reshape(patches, [tf.shape(images)[0], -1, patch_dims])
        return patches


class PatchEncoder(layers.Layer):
    def __init__(self, num_patches, projection_dim):
        super().__init__()
        self.projection = layers.Dense(projection_dim)
        self.position_embedding = layers.Embedding(
            input_dim=num_patches,
            output_dim=projection_dim
        )

    def call(self, patches):
        positions = tf.range(start=0, limit=tf.shape(patches)[1], delta=1)
        return self.projection(patches) + self.position_embedding(positions)

In [10]:
def build_model():
    inputs = layers.Input(shape=input_shape)

    # -------- Normalize --------
    x = layers.Rescaling(1./255)(inputs)

    # -------- CNN (ResNet50) --------
    base_model = ResNet50(
        weights='imagenet',
        include_top=False,
        input_tensor=x
    )
    base_model.trainable = False

    x_cnn = layers.GlobalAveragePooling2D()(base_model.output)

    # -------- ViT --------
    num_patches = (input_shape[0] // patch_size) ** 2

    patches = Patches(patch_size)(x)
    encoded = PatchEncoder(num_patches, projection_dim)(patches)

    for _ in range(transformer_layers):
        x1 = layers.LayerNormalization()(encoded)
        attn = layers.MultiHeadAttention(
            num_heads=num_heads,
            key_dim=projection_dim
        )(x1, x1)

        x2 = layers.Add()([attn, encoded])
        x3 = layers.LayerNormalization()(x2)
        encoded = layers.Add()([x3, x2])

    # -------- FIXED ViT → GRU --------
    x_vit = layers.LayerNormalization()(encoded)
    x_vit = layers.Flatten()(x_vit)

    # ✅ MUST: fixed dimension
    x_vit = layers.Dense(256, activation="relu")(x_vit)

    # ✅ fixed reshape (NO -1)
    x_vit = layers.Reshape((1, 256))(x_vit)

    x_vit = layers.GRU(128)(x_vit)

    # -------- Merge --------
    x = layers.concatenate([x_cnn, x_vit])
    x = layers.Dense(128, activation='relu')(x)
    x = layers.Dropout(0.3)(x)

    outputs = layers.Dense(num_classes, activation='softmax')(x)

    model = keras.Model(inputs, outputs)
    return model, base_model

In [11]:
model, base_model = build_model()

model.compile(
    optimizer=keras.optimizers.Adam(1e-4),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

model.summary()

ValueError: Shapes used to initialize variables must be fully-defined (no `None` dimensions). Received: shape=(None, 256) for variable path='dense_2/kernel'

In [ ]:
history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=epochs
)

In [ ]:
def get_gradcam(model, img_array, last_conv="conv5_block3_out"):
    grad_model = tf.keras.models.Model(
        model.inputs,
        [model.get_layer(last_conv).output, model.output]
    )

    with tf.GradientTape() as tape:
        conv_outputs, predictions = grad_model(img_array)
        class_idx = tf.argmax(predictions[0])
        loss = predictions[:, class_idx]

    grads = tape.gradient(loss, conv_outputs)
    pooled_grads = tf.reduce_mean(grads, axis=(0,1,2))

    conv_outputs = conv_outputs[0]
    heatmap = conv_outputs @ pooled_grads[..., tf.newaxis]
    heatmap = tf.squeeze(heatmap)

    heatmap = tf.maximum(heatmap,0) / tf.reduce_max(heatmap)
    return heatmap.numpy()

In [ ]:
def show_gradcam(model, image):
    img = image.numpy()
    img_input = tf.expand_dims(image, axis=0)

    heatmap = get_gradcam(model, img_input)
    heatmap = cv2.resize(heatmap, (img.shape[1], img.shape[0]))

    overlay = cv2.applyColorMap(np.uint8(255*heatmap), cv2.COLORMAP_JET)
    superimposed = overlay * 0.4 + img

    plt.imshow(superimposed.astype("uint8"))
    plt.title("Grad-CAM")
    plt.axis("off")
    plt.show()

In [ ]:
def get_vit_attention_map(model, image):
    image = tf.expand_dims(image, axis=0)

    for layer in model.layers:
        if isinstance(layer, tf.keras.layers.MultiHeadAttention):
            _, attn = layer(image, image, return_attention_scores=True)
            return tf.reduce_mean(attn, axis=1)[0].numpy()

In [ ]:
def show_vit(model, image):
    img = image.numpy()
    attn = get_vit_attention_map(model, image)

    size = int(np.sqrt(attn.shape[-1]))
    attn = attn.reshape((size,size))
    attn = cv2.resize(attn, (img.shape[1], img.shape[0]))

    plt.imshow(img.astype("uint8"))
    plt.imshow(attn, cmap="jet", alpha=0.4)
    plt.title("ViT Attention")
    plt.axis("off")
    plt.show()

In [ ]:
def show_shap(model, image):
    img_input = tf.expand_dims(image, axis=0)

    explainer = shap.GradientExplainer(model, img_input)
    shap_values = explainer.shap_values(img_input)

    shap.image_plot(shap_values, img_input.numpy())

In [ ]:
for images, labels in train_ds.take(1):
    img = images[0]

    show_gradcam(model, img)
    show_vit(model, img)
    show_shap(model, img)